# Data Cleaning Notebook

## Objective
Clean and preprocess the World Real-Time Food Prices dataset for further analysis.

## Dataset
- **Source**: `data/raw/WLD_RTFP_country_2023-10-02.csv`
- **Output**: `data/cleaned/food_prices_cleaned.csv`

## 1. Import Libraries

In [10]:
import pandas as pd
import numpy as np
import os

## 2. Load Raw Data

In [11]:
# Load the raw dataset
raw_data_path = '../data/raw/WLD_RTFP_country_2023-10-02.csv'
df = pd.read_csv(raw_data_path)

print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")

Dataset loaded successfully!
Shape: 4798 rows, 8 columns


## 3. Initial Data Exploration

In [12]:
# Display first few rows
df.head(10)

,Open,High,Low,Close,Inflation,country,ISO3,date
0,0.53,0.54,0.53,0.53,NaN,Afghanistan,AFG,2007-01-01
1,0.53,0.54,0.53,0.53,NaN,Afghanistan,AFG,2007-02-01
2,0.54,0.54,0.53,0.53,NaN,Afghanistan,AFG,2007-03-01
3,0.53,0.55,0.53,0.55,NaN,Afghanistan,AFG,2007-04-01
4,0.56,0.57,0.56,0.57,NaN,Afghanistan,AFG,2007-05-01
5,0.59,0.60,0.58,0.58,NaN,Afghanistan,AFG,2007-06-01
6,0.59,0.60,0.59,0.59,NaN,Afghanistan,AFG,2007-07-01
7,0.60,0.60,0.59,0.59,NaN,Afghanistan,AFG,2007-08-01
8,0.60,0.62,0.59,0.62,NaN,Afghanistan,AFG,2007-09-01
9,0.64,0.64,0.63,0.63,NaN,Afghanistan,AFG,2007-10-01


In [13]:
# Data types and info
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4798 entries, 0 to 4797
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Open       4734 non-null   float64
 1   High       4734 non-null   float64
 2   Low        4734 non-null   float64
 3   Close      4734 non-null   float64
 4   Inflation  4434 non-null   float64
 5   country    4798 non-null   str    
 6   ISO3       4798 non-null   str    
 7   date       4798 non-null   str    
dtypes: float64(5), str(3)
memory usage: 300.0 KB


In [14]:
# Statistical summary
df.describe()

,Open,High,Low,Close,Inflation
count,4734.000000,4734.000000,4734.000000,4734.000000,4434.000000
mean,1.491880,1.536158,1.451056,1.492398,14.692346
std,4.652457,4.883312,4.439229,4.633321,35.910342
min,0.010000,0.010000,0.010000,0.010000,-31.470000
25%,0.740000,0.750000,0.720000,0.740000,-0.487500
50%,0.960000,0.980000,0.950000,0.960000,5.360000
75%,1.100000,1.120000,1.077500,1.100000,16.372500
max,102.460000,106.480000,94.420000,94.420000,363.100000


In [15]:
# Check unique countries
print(f"Number of unique countries: {df['country'].nunique()}")
print(f"\nCountries: {df['country'].unique()}")

Number of unique countries: 25

Countries: <StringArray>
[             'Afghanistan',                  'Burundi',
             'Burkina Faso', 'Central African Republic',
                 'Cameroon',         'Congo, Dem. Rep.',
              'Congo, Rep.',              'Gambia, The',
            'Guinea-Bissau',                    'Haiti',
                     'Iraq',                  'Lao PDR',
                  'Lebanon',                  'Liberia',
                     'Mali',                  'Myanmar',
               'Mozambique',                    'Niger',
                  'Nigeria',                    'Sudan',
                  'Somalia',              'South Sudan',
     'Syrian Arab Republic',                     'Chad',
              'Yemen, Rep.']
Length: 25, dtype: str


## 4. Check for Missing Values

In [16]:
# Count missing values per column
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage (%)': missing_percentage.round(2)
})

print("Missing Values Analysis:")
missing_df

Missing Values Analysis:


,Missing Values,Percentage (%)
Open,64,1.33
High,64,1.33
Low,64,1.33
Close,64,1.33
Inflation,364,7.59
country,0,0.00
ISO3,0,0.00
date,0,0.00


## 5. Check for Duplicates

In [17]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

# Check for duplicate country-date combinations
duplicate_country_date = df.duplicated(subset=['country', 'date']).sum()
print(f"Duplicate country-date combinations: {duplicate_country_date}")

Number of duplicate rows: 0
Duplicate country-date combinations: 0


## 6. Data Cleaning Steps

### 6.1 Convert Date Column to DateTime

In [18]:
# Convert date column to datetime
df['date'] = pd.to_datetime(df['date'])

# Extract useful date components
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.month_name()

print("Date range:")
print(f"From: {df['date'].min()}")
print(f"To: {df['date'].max()}")

Date range:
From: 2007-01-01 00:00:00
To: 2023-10-01 00:00:00


### 6.2 Handle Missing Values in Inflation Column

In [19]:
# Check missing inflation values by country
missing_inflation_by_country = df[df['Inflation'].isnull()].groupby('country').size()
print("Missing Inflation values by country:")
print(missing_inflation_by_country)

Missing Inflation values by country:
country
Afghanistan                 12
Burkina Faso                12
Burundi                     12
Cameroon                    51
Central African Republic    12
Chad                        12
Congo, Dem. Rep.            12
Congo, Rep.                 12
Gambia, The                 12
Guinea-Bissau               12
Haiti                       12
Iraq                        12
Lao PDR                     37
Lebanon                     12
Liberia                     12
Mali                        12
Mozambique                  12
Myanmar                     12
Niger                       12
Nigeria                     12
Somalia                     12
South Sudan                 12
Sudan                       12
Syrian Arab Republic        12
Yemen, Rep.                 12
dtype: int64


In [20]:
# Option 1: Keep missing values as NaN (for analysis where we handle missing data)
# Option 2: Fill missing inflation with 0 (if inflation data not available)
# Option 3: Forward fill within each country group

# We'll create a cleaned version with NaN preserved and add a flag column
df['inflation_available'] = df['Inflation'].notna()

print(f"Records with inflation data: {df['inflation_available'].sum()}")
print(f"Records without inflation data: {(~df['inflation_available']).sum()}")

Records with inflation data: 4434
Records without inflation data: 364


### 6.3 Remove Duplicate Rows (if any)

In [21]:
# Remove exact duplicate rows
initial_rows = len(df)
df = df.drop_duplicates()
rows_removed = initial_rows - len(df)
print(f"Duplicate rows removed: {rows_removed}")

Duplicate rows removed: 0


### 6.4 Validate Numeric Columns

In [22]:
# Check for negative values in price columns (Open, High, Low, Close)
price_columns = ['Open', 'High', 'Low', 'Close']

for col in price_columns:
    negative_count = (df[col] < 0).sum()
    if negative_count > 0:
        print(f"Warning: {col} has {negative_count} negative values")
    else:
        print(f"{col}: No negative values (OK)")

Open: No negative values (OK)
High: No negative values (OK)
Low: No negative values (OK)
Close: No negative values (OK)


In [23]:
# Check data consistency: High should be >= Low
inconsistent_high_low = df[df['High'] < df['Low']]
print(f"Rows where High < Low: {len(inconsistent_high_low)}")

if len(inconsistent_high_low) > 0:
    print("Inconsistent rows:")
    display(inconsistent_high_low)

Rows where High < Low: 0


### 6.5 Calculate Additional Features

In [24]:
# Calculate price range (volatility indicator)
df['price_range'] = df['High'] - df['Low']

# Calculate price change (Close - Open)
df['price_change'] = df['Close'] - df['Open']

# Calculate percentage change
df['price_change_pct'] = ((df['Close'] - df['Open']) / df['Open']) * 100

print("New features added: price_range, price_change, price_change_pct")

New features added: price_range, price_change, price_change_pct


### 6.6 Rename Columns for Consistency

In [25]:
# Rename columns to snake_case for consistency
df = df.rename(columns={
    'Open': 'open',
    'High': 'high',
    'Low': 'low',
    'Close': 'close',
    'Inflation': 'inflation',
    'ISO3': 'iso3'
})

print("Columns renamed to snake_case")
print(f"Current columns: {df.columns.tolist()}")

Columns renamed to snake_case
Current columns: ['open', 'high', 'low', 'close', 'inflation', 'country', 'iso3', 'date', 'year', 'month', 'month_name', 'inflation_available', 'price_range', 'price_change', 'price_change_pct']


### 6.7 Reorder Columns

In [26]:
# Reorder columns logically
column_order = [
    'date', 'year', 'month', 'month_name',
    'country', 'iso3',
    'open', 'high', 'low', 'close',
    'price_range', 'price_change', 'price_change_pct',
    'inflation', 'inflation_available'
]

df = df[column_order]
print("Columns reordered")

Columns reordered


## 7. Final Data Validation

In [27]:
# Final check of cleaned data
print("=" * 50)
print("CLEANED DATA SUMMARY")
print("=" * 50)
print(f"\nShape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nDate Range: {df['date'].min()} to {df['date'].max()}")
print(f"\nCountries: {df['country'].nunique()}")
print(f"\nMissing Values:")
print(df.isnull().sum())

CLEANED DATA SUMMARY

Shape: 4798 rows, 15 columns

Date Range: 2007-01-01 00:00:00 to 2023-10-01 00:00:00

Countries: 25

Missing Values:
date                     0
year                     0
month                    0
month_name               0
country                  0
iso3                     0
open                    64
high                    64
low                     64
close                   64
price_range             64
price_change            64
price_change_pct        64
inflation              364
inflation_available      0
dtype: int64


In [28]:
# Preview cleaned data
df.head(10)

,date,year,month,month_name,country,iso3,open,high,low,close,price_range,price_change,price_change_pct,inflation,inflation_available
0,2007-01-01,2007,1,January,Afghanistan,AFG,0.53,0.54,0.53,0.53,0.01,0.00,0.000000,NaN,False
1,2007-02-01,2007,2,February,Afghanistan,AFG,0.53,0.54,0.53,0.53,0.01,0.00,0.000000,NaN,False
2,2007-03-01,2007,3,March,Afghanistan,AFG,0.54,0.54,0.53,0.53,0.01,-0.01,-1.851852,NaN,False
3,2007-04-01,2007,4,April,Afghanistan,AFG,0.53,0.55,0.53,0.55,0.02,0.02,3.773585,NaN,False
4,2007-05-01,2007,5,May,Afghanistan,AFG,0.56,0.57,0.56,0.57,0.01,0.01,1.785714,NaN,False
5,2007-06-01,2007,6,June,Afghanistan,AFG,0.59,0.60,0.58,0.58,0.02,-0.01,-1.694915,NaN,False
6,2007-07-01,2007,7,July,Afghanistan,AFG,0.59,0.60,0.59,0.59,0.01,0.00,0.000000,NaN,False
7,2007-08-01,2007,8,August,Afghanistan,AFG,0.60,0.60,0.59,0.59,0.01,-0.01,-1.666667,NaN,False
8,2007-09-01,2007,9,September,Afghanistan,AFG,0.60,0.62,0.59,0.62,0.03,0.02,3.333333,NaN,False
9,2007-10-01,2007,10,October,Afghanistan,AFG,0.64,0.64,0.63,0.63,0.01,-0.01,-1.562500,NaN,False


In [29]:
# Data types of cleaned dataset
df.dtypes

date                   datetime64[us]
year                            int32
month                           int32
month_name                        str
country                           str
iso3                              str
open                          float64
high                          float64
low                           float64
close                         float64
price_range                   float64
price_change                  float64
price_change_pct              float64
inflation                     float64
inflation_available              bool
dtype: object

## 8. Save Cleaned Data

In [30]:
# Create output directory if it doesn't exist
output_dir = '../data/cleaned'
os.makedirs(output_dir, exist_ok=True)

# Save cleaned data to CSV
output_path = os.path.join(output_dir, 'food_prices_cleaned.csv')
df.to_csv(output_path, index=False)

print(f"Cleaned data saved to: {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024:.2f} KB")

Cleaned data saved to: ../data/cleaned\food_prices_cleaned.csv
File size: 565.51 KB


## 9. Summary of Cleaning Steps

The following cleaning operations were performed:

1. **Loaded raw data** from CSV file
2. **Converted date column** to datetime format
3. **Extracted date components** (year, month, month_name)
4. **Handled missing values** - added flag for missing inflation data
5. **Removed duplicates** (if any)
6. **Validated numeric data** - checked for negative prices and data consistency
7. **Created new features** - price_range, price_change, price_change_pct
8. **Renamed columns** to snake_case for consistency
9. **Reordered columns** logically
10. **Saved cleaned data** to `data/cleaned/food_prices_cleaned.csv`